# TrustFed-IoT — Benchmark Experiments on GPU

**Before running:** Runtime → Change runtime type → **T4 GPU**

This notebook runs all comparison experiments (Proposed vs FedAvg vs Multi-Krum) across 5 attack scenarios with 100 FL rounds.

In [ ]:
#@title Step 1: Install dependencies
!pip install torch torchvision optuna requests matplotlib numpy tqdm

In [ ]:
#@title Step 2: Clone project from GitHub
#@markdown Replace YOUR_USERNAME with your GitHub username
import os

REPO_URL = "https://github.com/YOUR_USERNAME/trustfed-iot.git"  #@param {type:"string"}

!git clone $REPO_URL
PROJECT_DIR = REPO_URL.split("/")[-1].replace(".git", "")
os.chdir(PROJECT_DIR)
print(f"Switched to: {os.getcwd()}")

In [ ]:
#@title Step 3: Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU")

In [ ]:
#@title Step 4: Rebuild partition cache (validation split)
!python -m data.build_partition_cache

---
## Run Experiments

Choose ONE of the options below:
- **Option A:** Single shard (all jobs on this machine) — ~6 hours
- **Option B:** Shard 0 of 2 (run on 2 Colabs in parallel) — ~3 hours each

In [ ]:
#@title Step 5: Run benchmark experiments
#@markdown Choose execution mode:
MODE = "shard_0_of_2"  #@param ["single", "shard_0_of_2", "shard_1_of_2"]

if MODE == "single":
    !python -m experiments.run_all_experiments \
      --rounds 100 \
      --seeds 1 2 3 4 5 \
      --attacks clean gaussian sign_flip scaling label_flip \
      --methods proposed fedavg multikrum \
      --export-zip

elif MODE == "shard_0_of_2":
    !python -m experiments.run_all_experiments \
      --rounds 100 \
      --seeds 1 2 3 4 5 \
      --attacks clean gaussian sign_flip scaling label_flip \
      --methods proposed fedavg multikrum \
      --num-shards 2 --shard-index 0 \
      --export-zip

elif MODE == "shard_1_of_2":
    !python -m experiments.run_all_experiments \
      --rounds 100 \
      --seeds 1 2 3 4 5 \
      --attacks clean gaussian sign_flip scaling label_flip \
      --methods proposed fedavg multikrum \
      --num-shards 2 --shard-index 1 \
      --export-zip

In [ ]:
#@title Step 6: Generate plots (only if single mode or after merging shards)
#@markdown Skip this if you ran shard mode — merge first, then plot.
!python -m experiments.plot_experiments --root results/benchmark/

In [ ]:
#@title Step 7: Download results
!zip -r benchmark_results.zip results/benchmark/ results/exports/
from google.colab.files import download
download('benchmark_results.zip')